In [1]:
import pandas as pd 
import matplotlib.pyplot as plt

#clean data load 
df = pd.read_csv('../data/analysis_ready_data.csv')

#date ko formate me kare 
df['date'] = pd.to_datetime(df['date'])

print(f"Total rows: {len(df)}")
display(df.head())

Total rows: 36550


,date,sku_id,units_sold,revenue,unit_price,promo_flag,product_name,category,subcategory,launch_date,...,year,month,quarter,week,day_of_week,is_weekend,season,holiday,is_holiday,promotion_event
0,2024-01-01,SKU001,5,18320.25,3664.05,0,Product 001,Furniture,Chair,2022-04-09,...,2024,1,Q1,1,Monday,0,Winter,NaN,0,NaN
1,2024-01-01,SKU002,15,57085.35,3805.69,0,Product 002,Home Decor,Table,2024-05-01,...,2024,1,Q1,1,Monday,0,Winter,NaN,0,NaN
2,2024-01-01,SKU003,5,40391.15,8078.23,0,Product 003,Kitchen,Cushion,2023-12-22,...,2024,1,Q1,1,Monday,0,Winter,NaN,0,NaN
3,2024-01-01,SKU004,5,34307.85,6861.57,0,Product 004,Lighting,Cookware,2023-04-28,...,2024,1,Q1,1,Monday,0,Winter,NaN,0,NaN
4,2024-01-01,SKU005,12,113918.64,9493.22,0,Product 005,Storage,Lamp,2023-04-22,...,2024,1,Q1,1,Monday,0,Winter,NaN,0,NaN


In [2]:
import numpy as np

# 1. Create the WAPE function to measure forecast error
def calculate_wape(actual, forecast):
    return np.sum(np.abs(actual - forecast)) / np.sum(actual)

print("WAPE function is ready!")

# 2. Build the Seasonal-Naive Baseline Model
# First, sort the data by product (SKU) and date to ensure chronological order
df = df.sort_values(['sku_id', 'date'])

# Create a forecast by shifting the actual sales by 7 days (1 week lag)
df['baseline_forecast_7d'] = df.groupby('sku_id')['units_sold'].shift(7)

# Remove the first 7 days for each SKU since they won't have a past 7-day record (NaN values)
baseline_df = df.dropna(subset=['baseline_forecast_7d'])

# 3. Calculate the error of this baseline model
baseline_wape = calculate_wape(baseline_df['units_sold'], baseline_df['baseline_forecast_7d'])

print(f"Baseline WAPE (7-day Naive): {baseline_wape:.2%}")

WAPE function is ready!
Baseline WAPE (7-day Naive): 28.83%


In [4]:
print("Starting Feature Engineering...")

# Ensure data is sorted by SKU and Date
df = df.sort_values(['sku_id', 'date'])

# 1. Date Features 
df['day_of_week'] = df['date'].dt.dayofweek
df['month'] = df['date'].dt.month
df['is_weekend'] = df['day_of_week'].isin([5, 6]).astype(int)

# 2. Lag Features 
df['lag_1d'] = df.groupby('sku_id')['units_sold'].shift(1)
df['lag_2d'] = df.groupby('sku_id')['units_sold'].shift(2)
df['lag_3d'] = df.groupby('sku_id')['units_sold'].shift(3)

# 3. Rolling Average Features 
df['rolling_mean_7d'] = df.groupby('sku_id')['units_sold'].transform(
    lambda x: x.rolling(window=7, min_periods=1).mean()
)

# 4. Safely remove ONLY the rows where our new lag features are missing
ml_df = df.dropna(subset=['lag_1d', 'lag_2d', 'lag_3d'])

print("Feature Engineering Complete! New columns added.")
print(f"Rows ready for Machine Learning: {len(ml_df)}")

# View a small sample of the new features
display(ml_df[['date', 'sku_id', 'units_sold', 'day_of_week', 'lag_1d', 'rolling_mean_7d']].head())

Starting Feature Engineering...
Feature Engineering Complete! New columns added.
Rows ready for Machine Learning: 36400


,date,sku_id,units_sold,day_of_week,lag_1d,rolling_mean_7d
150,2024-01-04,SKU001,23,3,12.0,13.250000
200,2024-01-05,SKU001,16,4,23.0,13.800000
250,2024-01-06,SKU001,18,5,16.0,14.500000
300,2024-01-07,SKU001,19,6,18.0,15.142857
350,2024-01-08,SKU001,12,0,19.0,16.142857


In [5]:
from sklearn.ensemble import RandomForestRegressor
import numpy as np

print("Preparing Data for Machine Learning...")

# 1. Select the features the model will learn from
features = [
    'day_of_week', 'month', 'is_weekend', 
    'lag_1d', 'lag_2d', 'lag_3d', 
    'rolling_mean_7d', 'promo_flag', 'unit_price'
]

# Ensure there are no remaining missing values that could crash the model
ml_df = ml_df.fillna(0)

X = ml_df[features]
y = ml_df['units_sold']

# 2. Train/Test Split (Chronological split is required for time-series)
# We will train on the older 80% of data and test on the newest 20%
split_index = int(len(ml_df) * 0.8)

X_train, X_test = X.iloc[:split_index], X.iloc[split_index:]
y_train, y_test = y.iloc[:split_index], y.iloc[split_index:]

print(f"Training on {len(X_train)} rows, Testing on {len(X_test)} rows.")

# 3. Train the Model
print("Training Random Forest Model... (This might take 10-20 seconds)")
model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)

# 4. Make Predictions on the Test Set
predictions = model.predict(X_test)

# 5. Evaluate Accuracy using WAPE
def calculate_wape(actual, forecast):
    return np.sum(np.abs(actual - forecast)) / np.sum(actual)

ml_wape = calculate_wape(y_test.values, predictions)

print("-" * 30)
print(f"🎯 Baseline WAPE to beat: 28.83%")
print(f"✅ Machine Learning WAPE: {ml_wape:.2%}")
print("-" * 30)

Preparing Data for Machine Learning...
Training on 29120 rows, Testing on 7280 rows.
Training Random Forest Model... (This might take 10-20 seconds)
------------------------------
🎯 Baseline WAPE to beat: 28.83%
✅ Machine Learning WAPE: 17.85%
------------------------------


In [6]:
import joblib
import os

# 1. Models ko save karne ke liye ek naya folder banayein (agar pehle se nahi hai)
os.makedirs('../models', exist_ok=True)

# 2. Apne trained 'model' ko .pkl format mein save karein
joblib.dump(model, '../models/rf_demand_model.pkl')

print("✅ Model successfully saved! Location: ../models/rf_demand_model.pkl")

✅ Model successfully saved! Location: ../models/rf_demand_model.pkl
